# 2009 BD: measuring solar radiation pressure on a few-meter asteroid

2009 BD is a few-meter near-Earth asteroid on an Earth-like orbit (2.9 to 4 m across, depending on the albedo solution; Mommert et al. 2014) and one of the first asteroids for which solar radiation pressure was detected directly from ground-based astrometry (Micheli, Tholen & Elliott 2012, *New Astronomy* 17, 446). Sunlight pushes it measurably over a multi-year arc, so measuring the push weighs the object as an area-to-mass ratio.

What it does:

1. Queries the Minor Planet Center (MPC) for 2009 BD's optical astrometry, keeps the observations between two named filters (JPL's first and last night) and fits a seed orbit from a cold start.
2. Refines the orbit with the radial and transverse Marsden terms A1 and A2 as free parameters, the set JPL solves, from wide priors and not from JPL's solution.
3. Reports A1 in JPL's units beside the area-to-mass ratio it implies under the engine's radiation-pressure constant, against JPL's solution and the 2012 detection.

## Setup

On Google Colab (or any fresh environment), the first
`empyrean.initialize()` call pays a one-time ~740 MB kernel
download.

In [1]:
%pip install -q empyrean==0.11.0rc1

In [2]:
import empyrean
import numpy as np
from empyrean import (
    CartesianOrbits,
    CovarianceKind,
    JointFunctional,
    JointProvenance,
    JointQuality,
    NonGravParams,
    ODConfig,
    PackedJoint,
    PackedJoints,
    SolveFor,
    SolveForParams,
)

# Two named filters on the MPC record, which is queried live below. Each keeps
# the observations at or after / at or before a UTC time: JPL's first and last
# night, so a row the MPC adds outside the arc cannot move the numbers.
FIRST_NIGHT = "2009-01-16"
LAST_NIGHT = "2011-06-20T23:59"
# The engine's radiation-pressure constant: 4.56e-6 N/m^2 at 1 au (about 1367
# W/m^2 / c). One m^2/kg of area-to-mass ratio is this many au/d^2 of radial
# acceleration at 1 au under the r^-2 law, JPL's A1 convention.
K_SRP = 4.56e-6 * 86400.0**2 / 1.495978707e11  # au/d^2 per m^2/kg
# JPL SBDB solution of 2022-12-20 (SBDB-published values, nothing derived by
# empyrean; 180 observations, 2009-01-16 to 2011-06-20): the Marsden radial and
# transverse terms with their 1 sigma, au/d^2. The law is r^-2: ALN = 1,
# R0 = 1 au, m = 2, n = 0, k = 0.
JPL_A1, JPL_A1_SIGMA = 5.604736985528152e-11, 8.508e-12
JPL_A2, JPL_A2_SIGMA = -1.14638955920583e-12, 8.37e-14
# Micheli, Tholen and Elliott 2012 (108 observations, radiation pressure only).
MICHELI_AMR, MICHELI_SIGMA = 2.97e-4, 0.33e-4  # m^2/kg
# Priors, wide on every axis and not JPL's solution: 1e-2 au and 1e-4 au/d on the
# state, 1e-9 and 1e-11 au/d^2 on A1 and A2 (about 20 and 10 times JPL's values).
# A3 is held at zero by a tight prior, as in JPL's solved set (A1 and A2): the
# engine's refine has no per-axis fixed/solved switch today, so its posterior
# is the prior and contributes nothing.
STATE_SIGMA = (1e-2, 1e-2, 1e-2, 1e-4, 1e-4, 1e-4)
A_SIGMA = (1e-9, 1e-11, 1e-14)  # A1, A2, A3


def sci(x, digits):
    return f"{x:.{digits}e}"


empyrean.initialize()

## 1. The record and a cold state-only fit

Query the MPC, keep the observations between the two named filters, and fit position and velocity with the engine's default weights. No JPL solution is used.

In [3]:
obs = empyrean.query_observations(["2009 BD"])
obs = obs.apply_mask([FIRST_NIGHT <= t <= LAST_NIGHT for t in obs.obs_time.to_pylist()])
times = obs.obs_time.to_pylist()
print(
    f"{len(obs)} optical observations from {len(set(obs.stn.to_pylist()))} "
    f"stations, {min(times)[:10]} to {max(times)[:10]}"
)
config = ODConfig(solve_for=SolveForParams.STATE_ONLY)
weighting = config.weighting
print(
    f"Weighting: the engine default, preset {weighting.preset.name} (station "
    f"floors) with nightly de-weighting within "
    f"{weighting.additional_layers[0].max_gap_days:.1f} d"
)
base = empyrean.determine(obs, config=config).single()
s0 = base.summary
print(
    f"State-only fit (the seed): rejected {s0.num_rejected} of the {s0.num_obs} "
    f"observations; chi2/dof {s0.reduced_chi2:.3f}  "
    f'RMS RA.cos(d) {s0.rms_ra_arcsec:.3f}" Dec {s0.rms_dec_arcsec:.3f}"  '
    f"on the {s0.num_selected} kept"
)

190 optical observations from 24 stations, 2009-01-16 to 2011-06-20
Weighting: the engine default, preset VFCC2017 (station floors) with nightly de-weighting within 0.5 d


State-only fit (the seed): rejected 68 of the 190 observations; chi2/dof 0.184  RMS RA.cos(d) 0.377" Dec 0.715"  on the 122 kept


## 2. JPL's solution, in its own units and as an area-to-mass ratio

JPL's published A1 and A2 (SBDB, solution of 2022-12-20), and A1 converted to an area-to-mass ratio with the engine's own constant K.

In [4]:
print(
    f"\nRadiation-pressure constant: K = {K_SRP:.5e} au/d^2 per m^2/kg "
    "(4.56e-6 N/m^2 at 1 au, the engine's)"
)
print(
    f"JPL A1 = {sci(JPL_A1, 4)} +/- {sci(JPL_A1_SIGMA, 2)} au/d^2 "
    f"= {sci(JPL_A1 / K_SRP, 4)} +/- {sci(JPL_A1_SIGMA / K_SRP, 2)} m^2/kg "
    "(SBDB solution 2022-12-20)"
)
print(
    f"JPL A2 = {sci(JPL_A2, 4)} +/- {sci(JPL_A2_SIGMA, 2)} au/d^2 "
    "(transverse, the Yarkovsky drift)"
)


Radiation-pressure constant: K = 2.27545e-07 au/d^2 per m^2/kg (4.56e-6 N/m^2 at 1 au, the engine's)
JPL A1 = 5.6047e-11 +/- 8.51e-12 au/d^2 = 2.4631e-04 +/- 3.74e-05 m^2/kg (SBDB solution 2022-12-20)
JPL A2 = -1.1464e-12 +/- 8.37e-14 au/d^2 (transverse, the Yarkovsky drift)


## 3. Fit A1 and A2 with the state

Wide priors on the state and on A1 and A2; A3 is held at zero by a tight prior, as in JPL's solved set.

In [5]:
var = np.zeros((9, 9))
var[:6, :6] = np.diag(np.square(STATE_SIGMA))
var[6:, 6:] = np.diag(np.square(A_SIGMA))
tri = [var[i, j] for i in range(9) for j in range(i + 1)]
prior = PackedJoint(
    layout_present=0b111,  # the Marsden A1, A2, A3 axes
    layout_considered=0,
    layout_marginalized=0,
    width=9,
    tri=np.asarray(tri, dtype=np.float64),
    kind=CovarianceKind.LINEAR,
    quality=JointQuality.POSITIVE_DEFINITE,
    cov_rank=0,
    cov_null_axes=0,
    cov_min_eig_corr=float("nan"),
    functional=JointFunctional.STATE,
    provenance=JointProvenance.DERIVED,
    asserted_source=None,
)
primed = CartesianOrbits.from_kwargs(
    orbit_id=base.orbit.orbit_id.to_pylist(),
    object_id=base.orbit.object_id.to_pylist(),
    coordinates=base.orbit.coordinates,
    non_grav=NonGravParams.from_kwargs(
        a1=[0.0],
        a2=[0.0],
        a3=[0.0],
        model=["marsden"],
        alpha=[1.0],
        r0=[1.0],
        m=[2.0],
        n=[0.0],
        k=[0.0],
    ),
    covariance=PackedJoints.from_joints([prior]),
)
fit = empyrean.refine(
    primed, obs, config=ODConfig(solve_for_flags=SolveFor(marsden="solved"))
)
sc = fit.solved_covariance
ng = fit.orbit.non_grav
a1 = float(ng.a1.to_numpy(zero_copy_only=False)[0])
a2 = float(ng.a2.to_numpy(zero_copy_only=False)[0])
a3 = float(ng.a3.to_numpy(zero_copy_only=False)[0])
slot = sc.marsden_slot
a1_sigma = float(np.sqrt(sc.matrix[slot, slot]))
a2_sigma = float(np.sqrt(sc.matrix[slot + 1, slot + 1]))
a3_sigma = float(np.sqrt(sc.matrix[slot + 2, slot + 2]))
s = fit.summary
print(
    f"\nFit with A1 and A2: converged {fit.converged}  "
    f"chi2/dof {s.reduced_chi2:.3f}  "
    f'RMS RA.cos(d) {s.rms_ra_arcsec:.3f}" Dec {s.rms_dec_arcsec:.3f}"  '
    f"({s.num_selected}/{s.num_obs} obs)"
)
print(
    f"Priors: A1 +/- {sci(A_SIGMA[0], 0)}, A2 +/- {sci(A_SIGMA[1], 0)}, "
    f"A3 +/- {sci(A_SIGMA[2], 0)} (the hold on A3) au/d^2; "
    f"the fit keeps {100 * (a1_sigma / A_SIGMA[0]) ** 2:.2f}% and "
    f"{100 * (a2_sigma / A_SIGMA[1]) ** 2:.2f}% of those priors' information"
)
amr, amr_sigma = a1 / K_SRP, a1_sigma / K_SRP
print(f"A1 = {sci(a1, 4)} +/- {sci(a1_sigma, 2)} au/d^2")
print(
    f"  = area-to-mass ratio {sci(amr, 4)} +/- {sci(amr_sigma, 2)} m^2/kg "
    f"({amr / amr_sigma:.1f} sigma), A1 / K"
)
print(f"A2 = {sci(a2, 4)} +/- {sci(a2_sigma, 2)} au/d^2")
print(
    f"A3 = {sci(a3, 2)} +/- {sci(a3_sigma, 2)} au/d^2 (its prior, not a fit: the "
    "engine's refine has no per-axis fixed switch)"
)
scale = np.sqrt(s.reduced_chi2)
print(
    f"  scaled by the residuals (x sqrt(chi2/dof)): A1 +/- {sci(a1_sigma * scale, 2)}"
    f", A2 +/- {sci(a2_sigma * scale, 2)} au/d^2"
)


Fit with A1 and A2: converged True  chi2/dof 0.174  RMS RA.cos(d) 0.408" Dec 0.553"  (190/190 obs)
Priors: A1 +/- 1e-09, A2 +/- 1e-11, A3 +/- 1e-14 (the hold on A3) au/d^2; the fit keeps 0.01% and 0.02% of those priors' information
A1 = 5.3508e-11 +/- 9.23e-12 au/d^2
  = area-to-mass ratio 2.3515e-04 +/- 4.05e-05 m^2/kg (5.8 sigma), A1 / K
A2 = -1.1598e-12 +/- 1.41e-13 au/d^2
A3 = -1.42e-17 +/- 1.00e-14 au/d^2 (its prior, not a fit: the engine's refine has no per-axis fixed switch)
  scaled by the residuals (x sqrt(chi2/dof)): A1 +/- 3.85e-12, A2 +/- 5.90e-14 au/d^2


## 4. Against JPL's solution and the 2012 detection

The differences in combined sigma, with the sigmas themselves beside JPL's.

In [6]:
for name, v, sg, j, js in (
    ("A1", a1, a1_sigma, JPL_A1, JPL_A1_SIGMA),
    ("A2", a2, a2_sigma, JPL_A2, JPL_A2_SIGMA),
):
    print(
        f"{name}: ours - JPL = {(v - j) / np.hypot(sg, js):+.2f} combined sigma; "
        f"JPL sigma {sci(js, 2)}, ours/JPL {sg / js:.1f}"
    )
print(
    f"Area-to-mass ratio: ours - Micheli 2012 ({sci(MICHELI_AMR, 2)} +/- "
    f"{sci(MICHELI_SIGMA, 2)} m^2/kg, radiation pressure only) = "
    f"{(amr - MICHELI_AMR) / np.hypot(amr_sigma, MICHELI_SIGMA):+.2f} combined sigma"
)

A1: ours - JPL = -0.20 combined sigma; JPL sigma 8.51e-12, ours/JPL 1.1
A2: ours - JPL = -0.08 combined sigma; JPL sigma 8.37e-14, ours/JPL 1.7
Area-to-mass ratio: ours - Micheli 2012 (2.97e-04 +/- 3.30e-05 m^2/kg, radiation pressure only) = -1.18 combined sigma
